# 05 · Experimento principal, perfil promedio y VSS

Este notebook cubre la **sección 5.3** de la práctica:

1. **Experimento principal (K=200):** resolver la ruta óptima estocástica $x^\star$ con Integer L-shaped, con un límite de 30 minutos de cómputo (sección 5.3.1).
2. **Perfil promedio:** construir $\bar\xi_K$ y resolver la aproximación determinista $x^{prom}$ como un MILP de un solo bloque de recurso (sección 5.3.2).
3. **Comparación in-sample:** evaluar ambas rutas sobre los mismos 200 escenarios (sin reoptimizar) y calcular el valor de la solución estocástica $VSS_K$ (sección 5.3.3).

In [1]:
import time
import json
import heapq
import numpy as np
import pandas as pd
import pulp
from scipy.optimize import linprog
from pathlib import Path

PROC_DIR = Path("../data/processed")
RESULTS_TABLES = Path("../results/tables")
RESULTS_LOGS = Path("../results/logs")
RESULTS_TABLES.mkdir(parents=True, exist_ok=True)
RESULTS_LOGS.mkdir(parents=True, exist_ok=True)

sites = pd.read_csv(PROC_DIR / "sites.csv")
arc_order = pd.read_csv(PROC_DIR / "arc_order.csv")
arc_costs = pd.read_csv(PROC_DIR / "arc_costs.csv")
scenarios = pd.read_parquet(PROC_DIR / "scenarios_K200.parquet")

K = len(scenarios)
V = sites["i"].tolist()
N = [i for i in V if i != 0]
A = list(zip(arc_order["i"], arc_order["j"]))
n = len(N)

c_ij = {(row.i, row.j): row.c_ij for row in arc_costs.itertuples()}

H = 390.0; o_bar = 30.0; c_OT = 1.50; c_EM = 6.00
alpha = {i: 1.0 for i in N}

demand_cost = pd.DataFrame([
    {"i": 1,  "d_i": 30, "c_out_i": 2.60},{"i": 2,  "d_i": 30, "c_out_i": 2.40},
    {"i": 3,  "d_i": 35, "c_out_i": 2.80},{"i": 4,  "d_i": 30, "c_out_i": 2.30},
    {"i": 5,  "d_i": 25, "c_out_i": 2.10},{"i": 6,  "d_i": 25, "c_out_i": 2.00},
    {"i": 7,  "d_i": 35, "c_out_i": 2.70},{"i": 8,  "d_i": 40, "c_out_i": 3.00},
    {"i": 9,  "d_i": 35, "c_out_i": 2.90},{"i": 10, "d_i": 25, "c_out_i": 2.20},
]).set_index("i")
d_i = demand_cost["d_i"].to_dict()
c_out_i = demand_cost["c_out_i"].to_dict()

xi = {s: {(i, j): scenarios.loc[s, f"xi_{i}_{j}"] for (i, j) in A} for s in range(K)}

n_x = len(A)
n_u = len(N)
idx_x = {a: k for k, a in enumerate(A)}
idx_u = {i: n_x + k for k, i in enumerate(N)}
idx_theta = n_x + n_u

print(f"K = {K} escenarios, |A| = {len(A)} arcos")


K = 200 escenarios, |A| = 110 arcos


In [2]:
# Detección de CBC nativo (necesario para los MILPs de esta sección: perfil promedio).
import shutil

candidate_paths = [shutil.which("cbc"), "/opt/homebrew/bin/cbc", "/usr/local/bin/cbc"]
cbc_system_path = next((p for p in candidate_paths if p and Path(p).exists()), None)

if cbc_system_path:
    SOLVER = pulp.COIN_CMD(msg=False, path=cbc_system_path)
    print(f"Usando CBC del sistema: {cbc_system_path}")
else:
    SOLVER = pulp.PULP_CBC_CMD(msg=False)
    print("Usando CBC empaquetado con PuLP.")


Usando CBC del sistema: /opt/homebrew/bin/cbc


## 1. Funciones del Integer L-shaped (reutilizadas y validadas en `04_lshaped.ipynb`)

Mismo algoritmo, maestro LP por nodo, subproblema de recurso, separación de cortes, ramificación, poda, y re-chequeo de nodos enteros. La única diferencia respecto al notebook 4 es el **criterio de parada**: aquí usamos un **límite de tiempo de 30 minutos**.

In [5]:
def solve_subproblem(x_fixed: dict, xi_s: dict):
    T_x = sum(xi_s[(i, j)] * x_fixed[(i, j)] for (i, j) in A)
    B = H - T_x
    c = [-c_out_i[i] for i in N] + [c_OT, c_EM]
    A_ub = [[alpha[i] for i in N] + [-1, -1]]
    b_ub = [B]
    bounds = [(0, d_i[i]) for i in N] + [(0, o_bar), (0, None)]
    res = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method="highs")
    const = sum(c_out_i[i] * d_i[i] for i in N)
    obj = const + res.fun
    dual = res.ineqlin.marginals[0]
    return obj, dual


def evaluate_QK(x_fixed: dict, scenario_dict: dict, K_eval: int):
    """Generaliza evaluate_QK para poder reutilizarla también en la sección 3
    (comparación in-sample), donde se evalúa sobre los mismos K=200 escenarios
    para AMBAS rutas, sin reoptimizar."""
    Q_vals = np.zeros(K_eval)
    duals = np.zeros(K_eval)
    for s in range(K_eval):
        obj, dual = solve_subproblem(x_fixed, scenario_dict[s])
        Q_vals[s] = obj
        duals[s] = dual
    QK_val = Q_vals.mean()
    g = {}
    for (i, j) in A:
        g[(i, j)] = -np.mean([duals[s] * scenario_dict[s][(i, j)] for s in range(K_eval)])
    const_term = QK_val - sum(g[(i, j)] * x_fixed[(i, j)] for (i, j) in A)
    return QK_val, g, const_term


def build_and_solve_master(bounds: dict, std_cuts: list, int_cuts: list):
    n_vars = n_x + n_u + 1
    c = np.zeros(n_vars)
    for a in A:
        c[idx_x[a]] = c_ij[a]
    c[idx_theta] = 1.0

    var_bounds = []
    for a in A:
        lb, ub = bounds.get(a, (0, 1))
        var_bounds.append((lb, ub))
    for i in N:
        var_bounds.append((1, n))
    var_bounds.append((0, None))

    A_eq, b_eq = [], []
    for k in V:
        row = np.zeros(n_vars)
        for (i, j) in A:
            if i == k:
                row[idx_x[(i, j)]] += 1
        A_eq.append(row); b_eq.append(1.0)
        row = np.zeros(n_vars)
        for (i, j) in A:
            if j == k:
                row[idx_x[(i, j)]] += 1
        A_eq.append(row); b_eq.append(1.0)

    A_ub, b_ub = [], []
    for (i, j) in A:
        if i in N and j in N:
            row = np.zeros(n_vars)
            row[idx_u[i]] = 1; row[idx_u[j]] = -1; row[idx_x[(i, j)]] = n
            A_ub.append(row); b_ub.append(n - 1)

    for (g, const) in std_cuts:
        row = np.zeros(n_vars)
        for a in A:
            row[idx_x[a]] = g[a]
        row[idx_theta] = -1
        A_ub.append(row); b_ub.append(-const)

    for (coeffs, rhs) in int_cuts:
        row = np.zeros(n_vars)
        for a in A:
            row[idx_x[a]] = coeffs[a]
        row[idx_theta] = -1
        A_ub.append(row); b_ub.append(-rhs)

    res = linprog(c, A_ub=np.array(A_ub), b_ub=np.array(b_ub),
                   A_eq=np.array(A_eq), b_eq=np.array(b_eq),
                   bounds=var_bounds, method="highs")
    if not res.success:
        return None
    xbar = {a: res.x[idx_x[a]] for a in A}
    return xbar, res.x[idx_theta], res.fun


def make_integer_cut(xbar_binary: dict, QK_val: float, L: float = 0.0):
    S1 = [a for a in A if xbar_binary[a] > 0.5]
    coeffs = {a: (QK_val - L) if a in S1 else -(QK_val - L) for a in A}
    rhs = L - (QK_val - L) * (len(S1) - 1)
    return coeffs, rhs


def is_integral(xbar: dict, tol: float = 1e-6) -> bool:
    return all(abs(v - round(v)) < tol for v in xbar.values())


def round_binary(xbar: dict) -> dict:
    return {a: round(v) for a, v in xbar.items()}


def solve_lshaped(scenario_dict: dict, K_run: int, time_limit_sec: float, gap_tol: float = 1e-3):
    """Corre el Integer L-shaped completo con límite de TIEMPO (no de iteraciones)."""
    t_start = time.time()
    std_cuts, int_cuts = [], []
    UB = float("inf")
    best_x = None
    history = []
    counter = 0
    open_nodes = [(0.0, counter, {})]
    heapq.heapify(open_nodes)
    it = 0
    gap = float("inf")
    LB = 0.0
    timed_out = False

    while open_nodes:
        if time.time() - t_start > time_limit_sec:
            timed_out = True
            break
        it += 1
        bound_est, nid, bounds = heapq.heappop(open_nodes)

        elapsed = time.time() - t_start
        print("\n" + "-" * 70)
        print(f"ITERACIÓN PRINCIPAL {it}")
        print(f"Tiempo transcurrido : {elapsed:.2f} s")
        print(f"Nodo                : {nid}")
        print(f"Bound estimado      : {bound_est:.4f}")
        print(f"Nodos abiertos      : {len(open_nodes)}")
        print(f"UB actual           : {UB:.4f}" if UB < float("inf") else "UB actual           : inf")
        print(f"LB actual           : {LB:.4f}")
        print(f"Standard cuts       : {len(std_cuts)}")
        print(f"Integer cuts        : {len(int_cuts)}")

        if bound_est >= UB - 1e-9:
            continue

        node_exhausted = False
        same_node_round = 0
        while not node_exhausted and same_node_round < 50:
            same_node_round += 1
            if time.time() - t_start > time_limit_sec:
                timed_out = True
                break

            result = build_and_solve_master(bounds, std_cuts, int_cuts)
            if result is None:
                node_exhausted = True
                break
            xbar, theta_val, node_LB = result
            if node_LB >= UB - 1e-9:
                node_exhausted = True
                break

            for _ in range(30):
                QK_val, g, const = evaluate_QK(xbar, scenario_dict, K_run)
                if theta_val < QK_val - 1e-4:
                    std_cuts.append((g, const))
                    result = build_and_solve_master(bounds, std_cuts, int_cuts)
                    if result is None:
                        break
                    xbar, theta_val, node_LB = result
                    if node_LB >= UB - 1e-9:
                        break
                else:
                    break

            if result is None or node_LB >= UB - 1e-9:
                node_exhausted = True
                break

            if is_integral(xbar):
                xb = round_binary(xbar)
                QK_val, g, const = evaluate_QK(xb, scenario_dict, K_run)
                true_cost = sum(c_ij[a] * xb[a] for a in A) + QK_val
                if true_cost < UB - 1e-9:
                    UB = true_cost
                    best_x = xb
                int_cuts.append(make_integer_cut(xb, QK_val, L=0.0))
            else:
                frac = {a: v for a, v in xbar.items() if abs(v - round(v)) > 1e-6}
                branch_arc = min(frac, key=lambda a: abs(frac[a] - 0.5))
                for fix_val in (0, 1):
                    child_bounds = dict(bounds)
                    child_bounds[branch_arc] = (fix_val, fix_val)
                    counter += 1
                    heapq.heappush(open_nodes, (node_LB, counter, child_bounds))
                node_exhausted = True

        if timed_out:
            break

        LB = min([b for b, _, _ in open_nodes], default=UB)
        gap = (UB - LB) / max(1, abs(UB)) if UB < float("inf") else float("inf")
        history.append((it, LB, UB, gap, len(std_cuts) + len(int_cuts)))
        if gap <= gap_tol:
            break

    solve_time = time.time() - t_start
    return {
        "best_x": best_x, "UB": UB, "LB": LB, "gap": gap, "it": it,
        "n_std_cuts": len(std_cuts), "n_int_cuts": len(int_cuts),
        "solve_time": solve_time, "history": history, "timed_out": timed_out,
    }

print("Funciones cargadas.")


Funciones cargadas.


## 2. Experimento principal: resolver $x^\star$ con K=200

Límite de 30 minutos. Si no se alcanza la tolerancia de gap, se reporta la mejor ruta incumbente junto con LB, UB y la brecha final.

In [ ]:
TIME_LIMIT_SEC = 30 * 60  # 30 minutos, según sección 5.3.1

print(f"Iniciando Integer L-shaped con K={K}, límite de {TIME_LIMIT_SEC/60:.0f} minutos...")
result_main = solve_lshaped(xi, K, TIME_LIMIT_SEC, gap_tol=1e-3)

x_star = result_main["best_x"]
print(f"\nTerminado en {result_main['solve_time']:.1f}s ({result_main['solve_time']/60:.2f} min)")
print(f"¿Se agotó el tiempo?: {result_main['timed_out']}")
print(f"Iteraciones: {result_main['it']}")
print(f"UB (x*): {result_main['UB']:.4f}")
print(f"LB: {result_main['LB']:.4f}")
print(f"Gap final: {result_main['gap']:.6f}")
print(f"Cortes: {result_main['n_std_cuts']} estándar + {result_main['n_int_cuts']} enteros")

if result_main["timed_out"] and result_main["gap"] > 1e-3:
    print("   No se alcanzó la tolerancia de gap dentro del límite de tiempo.")
    print("   Se reporta la mejor ruta incumbente (x*) junto con LB, UB y la brecha,")
    print("   tal como lo permite explícitamente la sección 5.3.1 del enunciado.")


Iniciando Integer L-shaped con K=200, límite de 30 minutos...

----------------------------------------------------------------------
ITERACIÓN PRINCIPAL 1
Tiempo transcurrido : 0.00 s
Nodo                : 0
Bound estimado      : 0.0000
Nodos abiertos      : 0
UB actual           : inf
LB actual           : 0.0000
Standard cuts       : 0
Integer cuts        : 0

----------------------------------------------------------------------
ITERACIÓN PRINCIPAL 2
Tiempo transcurrido : 2.20 s
Nodo                : 1
Bound estimado      : 30.5389
Nodos abiertos      : 1
UB actual           : inf
LB actual           : 30.5389
Standard cuts       : 30
Integer cuts        : 0

----------------------------------------------------------------------
ITERACIÓN PRINCIPAL 3
Tiempo transcurrido : 3.47 s
Nodo                : 2
Bound estimado      : 30.5389
Nodos abiertos      : 2
UB actual           : inf
LB actual           : 30.5389
Standard cuts       : 48
Integer cuts        : 0

----------------------

In [7]:
selected_arcs = [a for a in A if x_star[a] > 0.5]
next_node = {i: j for (i, j) in selected_arcs}
route_star = [0]
current = 0
for _ in range(len(V) - 1):
    current = next_node[current]
    route_star.append(current)
route_star.append(0)

assert len(set(route_star[:-1])) == len(V), "La ruta x* no visita todos los nodos exactamente una vez."

route_names = sites.set_index("i")["name"]
print("Ruta óptima x* (K=200):")
print(" -> ".join(route_names[node] for node in route_star))


Ruta óptima x* (K=200):
Javits Center (Depósito) -> Madison Square Garden -> Times Square -> Rockefeller Center -> Grand Central Terminal -> New York Public Library -> Union Square -> Washington Square Park -> South Street Seaport (Pier 17) -> New York Stock Exchange -> One World Trade Center -> Javits Center (Depósito)


## 3. Perfil promedio $\bar\xi_K$ y ruta determinista $x^{prom}$

$$\bar\xi_{ij,K} = \frac{1}{K}\sum_{s=1}^K \xi_{ij}^{(s)}$$

Se resuelve la aproximación determinista como un **MILP con un único bloque de recurso**. Es estructuralmente igual a la forma extensa del notebook 03, pero con $K=1$ "escenario" (el vector promedio).

In [8]:
xi_bar = {a: np.mean([xi[s][a] for s in range(K)]) for a in A}

t0 = time.time()

prob = pulp.LpProblem("average_profile", pulp.LpMinimize)
x = pulp.LpVariable.dicts("x", A, cat="Binary")
u_order = pulp.LpVariable.dicts("u_order", N, lowBound=1, upBound=n, cat="Continuous")

for k in V:
    prob += pulp.lpSum(x[(k, j)] for (i, j) in A if i == k) == 1
    prob += pulp.lpSum(x[(i, k)] for (i, j) in A if j == k) == 1
for (i, j) in A:
    if i in N and j in N:
        prob += u_order[i] - u_order[j] + n * x[(i, j)] <= n - 1

w = pulp.LpVariable.dicts("w", N, lowBound=0)
r = pulp.LpVariable.dicts("r", N, lowBound=0)
o = pulp.LpVariable("o", lowBound=0, upBound=o_bar)
e = pulp.LpVariable("e", lowBound=0)

for i in N:
    prob += w[i] <= d_i[i]
    prob += w[i] + r[i] == d_i[i]

prob += (
    pulp.lpSum(xi_bar[(i, j)] * x[(i, j)] for (i, j) in A)
    + pulp.lpSum(alpha[i] * w[i] for i in N)
    <= H + o + e
)

first_stage_cost = pulp.lpSum(c_ij[(i, j)] * x[(i, j)] for (i, j) in A)
recourse_cost = pulp.lpSum(c_out_i[i] * r[i] for i in N) + c_OT * o + c_EM * e
prob += first_stage_cost + recourse_cost

n_vars_prom = prob.numVariables()
n_constrs_prom = prob.numConstraints()

prob.solve(SOLVER)
solve_time_prom = time.time() - t0

status_prom = pulp.LpStatus[prob.status]
obj_prom = pulp.value(prob.objective)

print(f"Estado: {status_prom}")
print(f"Variables: {n_vars_prom}, Restricciones: {n_constrs_prom}")
print(f"Objetivo (evaluado en xi_bar, NO comparable directo con Q_K): {obj_prom:.4f}")
print(f"Tiempo de solución: {solve_time_prom:.2f}s")


Estado: Optimal
Variables: 142, Restricciones: 133
Objetivo (evaluado en xi_bar, NO comparable directo con Q_K): 51.9286
Tiempo de solución: 0.20s


In [9]:
x_prom = {a: round(pulp.value(x[a])) for a in A}

selected_arcs_prom = [a for a in A if x_prom[a] > 0.5]
next_node_prom = {i: j for (i, j) in selected_arcs_prom}
route_prom = [0]
current = 0
for _ in range(len(V) - 1):
    current = next_node_prom[current]
    route_prom.append(current)
route_prom.append(0)

assert len(set(route_prom[:-1])) == len(V), "La ruta x_prom no visita todos los nodos exactamente una vez."

print("Ruta del perfil promedio x_prom:")
print(" -> ".join(route_names[node] for node in route_prom))

if route_star == route_prom:
    print("\n>>> x* y x_prom son EXACTAMENTE LA MISMA RUTA. <<<")
else:
    print("\nx* y x_prom son rutas DISTINTAS.")


Ruta del perfil promedio x_prom:
Javits Center (Depósito) -> Madison Square Garden -> Times Square -> Rockefeller Center -> Grand Central Terminal -> New York Public Library -> Union Square -> Washington Square Park -> South Street Seaport (Pier 17) -> New York Stock Exchange -> One World Trade Center -> Javits Center (Depósito)

>>> x* y x_prom son EXACTAMENTE LA MISMA RUTA. <<<


## 4. Comparación in-sample: $VSS_K$

Nota: $\bar Q_K(x) = Q(x,\bar\xi_K)$ (el objetivo de la celda anterior) **no es lo mismo** que $Q_K(x) = \frac{1}{K}\sum_s Q(x,\xi^{(s)})$. Para comparar $x^\star$ y $x^{prom}$ de forma justa, ambas rutas se evalúan aquí, sobre los mismos 200 escenarios reales:

$$\hat C_K(x) := \sum_{(i,j)\in A} c_{ij}x_{ij} + \frac{1}{K}\sum_{s=1}^K Q(x,\xi^{(s)})$$

$$VSS_K := \hat C_K(x^{prom}) - \hat C_K(x^\star)$$

In [ ]:
QK_star, _, _ = evaluate_QK(x_star, xi, K)
QK_prom, _, _ = evaluate_QK(x_prom, xi, K)

C_hat_star = sum(c_ij[a] * x_star[a] for a in A) + QK_star
C_hat_prom = sum(c_ij[a] * x_prom[a] for a in A) + QK_prom

VSS_K = C_hat_prom - C_hat_star

print(f"C_hat_K(x*):    {C_hat_star:.4f}")
print(f"C_hat_K(x_prom): {C_hat_prom:.4f}")
print(f"VSS_K = C_hat_K(x_prom) - C_hat_K(x*) = {VSS_K:.4f}")

if not result_main["timed_out"] or result_main["gap"] <= 1e-3:
    print("\nx* fue certificada como óptima (gap <= 1e-3): se espera VSS_K >= 0 (salvo tolerancias numéricas).")
    if VSS_K < -1e-6:
        print(f"  ADVERTENCIA: VSS_K es negativo ({VSS_K:.6f}) a pesar de que x* fue certificada óptima. Revisar.")
    else:
        print(f"✓ VSS_K >= 0, consistente con la optimalidad certificada de x*.")
else:
    print("\nx* es solo la mejor incumbente (no se certificó optimalidad por límite de tiempo).")
    print("Se reporta el signo de VSS_K sin imponer la condición VSS_K >= 0.")


C_hat_K(x*):    53.4642
C_hat_K(x_prom): 53.4642
VSS_K = C_hat_K(x_prom) - C_hat_K(x*) = 0.0000

x* fue certificada como óptima (gap <= 1e-3): se espera VSS_K >= 0 (salvo tolerancias numéricas).
✓ VSS_K >= 0, consistente con la optimalidad certificada de x*.


## 5. Frecuencias de uso de recurso (diagnóstico adicional)

Para cada ruta, en cuántos de los 200 escenarios se activa tiempo adicional ordinario, tercerización y sobretiempo de emergencia. Este mismo cálculo se repetirá en el notebook 06 con los escenarios de febrero (sección 5.3.4), pero calcularlo aquí también ayuda a caracterizar el comportamiento in-sample de ambas rutas.

In [11]:
def resource_usage_frequencies(x_fixed, scenario_dict, K_eval):
    freq_o, freq_e, freq_r = 0, 0, 0
    for s in range(K_eval):
        T_x = sum(scenario_dict[s][a] * x_fixed[a] for a in A)
        B = H - T_x
        c = [-c_out_i[i] for i in N] + [c_OT, c_EM]
        A_ub = [[alpha[i] for i in N] + [-1, -1]]
        b_ub = [B]
        bounds = [(0, d_i[i]) for i in N] + [(0, o_bar), (0, None)]
        res = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method="highs")
        w_vals = res.x[:len(N)]
        o_val, e_val = res.x[len(N)], res.x[len(N)+1]
        if o_val > 1e-6:
            freq_o += 1
        if e_val > 1e-6:
            freq_e += 1
        if any((d_i[i] - w_vals[k]) > 1e-6 for k, i in enumerate(N)):
            freq_r += 1
    return freq_o / K_eval, freq_e / K_eval, freq_r / K_eval

freq_o_star, freq_e_star, freq_r_star = resource_usage_frequencies(x_star, xi, K)
freq_o_prom, freq_e_prom, freq_r_prom = resource_usage_frequencies(x_prom, xi, K)

print("Frecuencias de uso de recurso (in-sample, K=200):")
print(f"{'':20s} {'x*':>10s} {'x_prom':>10s}")
print(f"{'Tiempo adicional':20s} {freq_o_star:>10.1%} {freq_o_prom:>10.1%}")
print(f"{'Emergencia':20s} {freq_e_star:>10.1%} {freq_e_prom:>10.1%}")
print(f"{'Tercerización':20s} {freq_r_star:>10.1%} {freq_r_prom:>10.1%}")


Frecuencias de uso de recurso (in-sample, K=200):
                             x*     x_prom
Tiempo adicional          95.5%      95.5%
Emergencia                 0.0%       0.0%
Tercerización             22.5%      22.5%


## 6. Guardar resultados

Estos resultados (en particular `route_star` y `route_prom`, junto con `c_ij`) se reutilizan en `06_out_of_sample.ipynb` para la validación con datos de febrero.

In [12]:
result = {
    "K": K,
    "x_star": {f"{a[0]}_{a[1]}": v for a, v in x_star.items()},
    "route_star": route_star,
    "x_prom": {f"{a[0]}_{a[1]}": v for a, v in x_prom.items()},
    "route_prom": route_prom,
    "routes_identical": route_star == route_prom,

    "lshaped": {
        "UB": result_main["UB"], "LB": result_main["LB"], "gap": result_main["gap"],
        "n_iterations": result_main["it"], "solve_time_sec": result_main["solve_time"],
        "timed_out": result_main["timed_out"],
        "n_std_cuts": result_main["n_std_cuts"], "n_int_cuts": result_main["n_int_cuts"],
    },

    "average_profile": {
        "objective_at_xi_bar": obj_prom, "status": status_prom,
        "n_variables": n_vars_prom, "n_constraints": n_constrs_prom,
        "solve_time_sec": solve_time_prom,
    },

    "in_sample_comparison": {
        "C_hat_K_star": C_hat_star, "C_hat_K_prom": C_hat_prom, "VSS_K": VSS_K,
        "x_star_certified_optimal": (not result_main["timed_out"]) or (result_main["gap"] <= 1e-3),
    },

    "resource_usage_frequencies": {
        "x_star": {"overtime": freq_o_star, "emergency": freq_e_star, "outsourcing": freq_r_star},
        "x_prom": {"overtime": freq_o_prom, "emergency": freq_e_prom, "outsourcing": freq_r_prom},
    },
}

with open(PROC_DIR / "main_experiment_K200_results.json", "w") as f:
    json.dump(result, f, indent=2)

history_df = pd.DataFrame(result_main["history"], columns=["iteration", "LB", "UB", "gap", "n_cuts"])
history_df.to_csv(RESULTS_LOGS / "lshaped_convergence_K200.csv", index=False)

print(f"Guardado: {PROC_DIR / 'main_experiment_K200_results.json'}")
print(f"Guardado: {RESULTS_LOGS / 'lshaped_convergence_K200.csv'}")


Guardado: ../data/processed/main_experiment_K200_results.json
Guardado: ../results/logs/lshaped_convergence_K200.csv
